# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atif929/flyrank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My rule and its reason codes

My baseline rule identifies pages that are strong candidates for a content refresh. Pages receive higher scores when they have many impressions, few clicks, a low click-through rate (CTR), and a poor average search position. These signals suggest that a page is visible in search results but is not attracting enough clicks or ranking strongly.

**Reason Codes**

- REFRESH_LOW_CTR – The page has low CTR despite receiving impressions.
- REFRESH_LOW_POSITION – The page ranks poorly in search results.
- REFRESH_HIGH_PRIORITY – Multiple signals indicate the page should be reviewed first.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Baseline rule defined successfully.")


Baseline rule defined successfully.


## Build the ranked queue

I use a simple rule-based baseline score to rank pages for content refresh. The score increases when a page has high impressions, low clicks, low CTR, and a poor average search position. The output is a ranked queue with one action label and one reason code for each page. This baseline is intended for comparison with future machine learning models.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

# Load Hugging Face token
token = userdata.get("HF_TOKEN")

# Download March 2026 data
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

con = duckdb.connect()

df = con.execute(f"""
SELECT
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{file_path}')
LIMIT 100000
""").fetch_df()
# -------------------------------
# Build baseline queue
# -------------------------------

queue = df.copy()

# CTR
queue["ctr"] = queue["gsc_clicks"] / queue["gsc_impressions"].replace(0, 1)

# Baseline score
queue["baseline_score"] = (
    queue["gsc_impressions"] * 0.4 +
    (1 - queue["ctr"]) * 1000 +
    queue["gsc_avg_position"] * 5
)

# Reason codes
queue["reason_code"] = "REFRESH_LOW_CTR"

queue.loc[
    queue["gsc_avg_position"] > 20,
    "reason_code"
] = "REFRESH_LOW_POSITION"

queue.loc[
    (queue["gsc_avg_position"] > 20) &
    (queue["ctr"] < 0.03),
    "reason_code"
] = "REFRESH_HIGH_PRIORITY"

# Action
queue["action"] = "Review"

# Rank
queue = queue.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows:", len(queue))
display(
    queue[
        [
            "content_hash_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Rows: 100000


,content_hash_id,baseline_score,reason_code,action
90707,content_36e53e9c707674fc,4031.644187,REFRESH_HIGH_PRIORITY,Review
1352,content_fd2117c2c6790e4b,3779.146065,REFRESH_LOW_CTR,Review
8618,content_29c4a3831609805d,3371.200202,REFRESH_LOW_CTR,Review
9080,content_e8b074fd4a082388,2731.547148,REFRESH_LOW_CTR,Review
4102,content_cf651123f1085418,2484.850288,REFRESH_LOW_CTR,Review
1026,content_00d4fdf6e48a2d38,2458.923038,REFRESH_LOW_CTR,Review
95548,content_db1cf8cf217e6051,2431.670608,REFRESH_HIGH_PRIORITY,Review
91164,content_f7c9fcc26f6e23c1,2413.337122,REFRESH_LOW_CTR,Review
96070,content_99c63c59330193de,2347.127586,REFRESH_HIGH_PRIORITY,Review
14855,content_62673eea26c31c17,2343.334735,REFRESH_LOW_CTR,Review


## Top-20 Review

The ranked queue was reviewed manually to ensure the highest-scoring pages are reasonable refresh candidates. Each page receives one action, one reason code, a confidence note, and a brief explanation of what could make the recommendation incorrect. This review helps identify weaknesses in the baseline rule before building a machine learning model.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20).copy()

top20["confidence"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "External factors such as seasonality, recent content updates, or search algorithm changes."
)

review = top20[
    [
        "content_hash_id",
        "action",
        "reason_code",
        "confidence",
        "what_would_make_it_wrong"
    ]
]

display(review)

,content_hash_id,action,reason_code,confidence,what_would_make_it_wrong
90707,content_36e53e9c707674fc,Review,REFRESH_HIGH_PRIORITY,Medium,"External factors such as seasonality, recent c..."
1352,content_fd2117c2c6790e4b,Review,REFRESH_LOW_CTR,Medium,"External factors such as seasonality, recent c..."
8618,content_29c4a3831609805d,Review,REFRESH_LOW_CTR,Medium,"External factors such as seasonality, recent c..."
9080,content_e8b074fd4a082388,Review,REFRESH_LOW_CTR,Medium,"External factors such as seasonality, recent c..."
4102,content_cf651123f1085418,Review,REFRESH_LOW_CTR,Medium,"External factors such as seasonality, recent c..."
1026,content_00d4fdf6e48a2d38,Review,REFRESH_LOW_CTR,Medium,"External factors such as seasonality, recent c..."
95548,content_db1cf8cf217e6051,Review,REFRESH_HIGH_PRIORITY,Medium,"External factors such as seasonality, recent c..."
91164,content_f7c9fcc26f6e23c1,Review,REFRESH_LOW_CTR,Medium,"External factors such as seasonality, recent c..."
96070,content_99c63c59330193de,Review,REFRESH_HIGH_PRIORITY,Medium,"External factors such as seasonality, recent c..."
14855,content_62673eea26c31c17,Review,REFRESH_LOW_CTR,Medium,"External factors such as seasonality, recent c..."


## Weak picks + leakage check

Some high-ranked pages may not actually require a content refresh. For example, seasonal pages or recently updated content may temporarily have low CTR or poor rankings. The baseline rule cannot detect these situations.

No future information, product flags, or label-derived features were used when calculating the baseline score. The score relies only on observed search performance metrics that would have been available at the decision time, reducing the risk of data leakage.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Leakage Check")
print("- No future windows used.")
print("- No product flags used.")
print("- No label-derived features used.")
print("- Baseline score uses only current observed metrics.")

Leakage Check
- No future windows used.
- No product flags used.
- No label-derived features used.
- Baseline score uses only current observed metrics.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.